### MapReduce Connection & Setup

In [6]:
import pymongo
from pymongo import MongoClient
from bson.code import Code

MONGO_URI = "mongodb://localhost:27017"
client = MongoClient(MONGO_URI)

db = client["amazon_reviews_db"]
reviews_col = db["reviews"]

print("Connected to database successfully. BSON Code imported for MapReduce.")

Connected to database successfully. BSON Code imported for MapReduce.


### Job 1: Frequency (Review Count per Product)
**Objective:** Emulate a simple `GROUP BY` count to find how many reviews each product has.

**Map:** Emits the product ID (`asin`) as the key and a value of `1` for every document.

**Reduce:** Takes the array of `1`s for each key and sums them up using `Array.sum()`.

In [7]:
map_freq = Code("function() { emit(this.asin, 1); }")
reduce_freq = Code("function(key, values) { return Array.sum(values); }")

print("Executing Job 1: Frequency...")


result_freq_info = db.command(
    "mapReduce", "reviews",
    map=map_freq,
    reduce=reduce_freq,
    out="mr_frequency_results"
)

print("--- Job 1 Results (Sample of 5) ---")

for doc in db["mr_frequency_results"].find().limit(5):
    print(doc)

Executing Job 1: Frequency...
--- Job 1 Results (Sample of 5) ---
{'_id': 'B0160CP1GE', 'value': 1.0}
{'_id': 'B0064LWISQ', 'value': 1.0}
{'_id': 'B00XS423SC', 'value': 2.0}
{'_id': 'B005KEL4NI', 'value': 1.0}
{'_id': 'B017W4PA60', 'value': 1.0}


### Job 2: Aggregation (Total Helpful Votes per Product)
**Objective:** Calculate the total number of helpful votes a product's reviews have accumulated.

**Map:** Emits the product ID (`asin`) as the key and the `vote` integer as the value (defaulting to 0 if null).

**Reduce:** Sums the array of vote values for each product.

**Execution:** Uses `db.command("mapReduce", ...)` to bypass the deprecated PyMongo collection method, writing results to the `mr_votes_results` collection.

In [ ]:
map_votes = Code("function() { emit(this.asin, this.vote || 0); }")
reduce_votes = Code("function(key, values) { return Array.sum(values); }")

print("Executing Job 2: Aggregation (Votes)...")

result_votes_info = db.command(
    "mapReduce", "reviews", 
    map=map_votes, 
    reduce=reduce_votes, 
    out="mr_votes_results"
)

print("--- Job 2 Results (Sample of 5) ---")
for doc in db["mr_votes_results"].find().limit(5):
    print(doc)

Executing Job 2: Aggregation (Votes)...
--- Job 2 Results (Sample of 5) ---
{'_id': 'B005PVXI80', 'value': 0.0}
{'_id': 'B0088K7QUQ', 'value': 0.0}
{'_id': 'B000GIT002', 'value': 52.0}
{'_id': 'B00EB8BVVO', 'value': 0.0}
{'_id': 'B00XHXXXQA', 'value': 0.0}


### Job 3: Custom Logic (Average Helpful Votes per User's Reviews)
**Objective:** Determine user credibility by calculating how many helpful votes their reviews get on average.

**Map:** Emits the `reviewerID` as the key. The value is a complex object containing both their votes for that review and a count of 1.

**Reduce:** Iterates through the emitted objects, summing up both the `total_votes` and the `total_reviews` for that user.

**Finalize:** A post-processing step that takes the fully reduced object and divides total votes by total reviews to calculate the `avg_votes_per_review`.

**Execution:** Uses `db.command("mapReduce", ...)` and passes the finalize function as an additional argument, writing to `mr_custom_results`.

In [9]:
map_custom = Code("""
function() {
    emit(this.reviewerID, { total_votes: this.vote || 0, total_reviews: 1 });
}
""")

reduce_custom = Code("""
function(key, values) {
    var reducedVal = { total_votes: 0, total_reviews: 0 };
    for (var i = 0; i < values.length; i++) {
        reducedVal.total_votes += values[i].total_votes;
        reducedVal.total_reviews += values[i].total_reviews;
    }
    return reducedVal;
}
""")

finalize_custom = Code("""
function(key, reducedVal) {
    reducedVal.avg_votes_per_review = reducedVal.total_votes / reducedVal.total_reviews;
    return reducedVal;
}
""")

print("Executing Job 3: Custom Logic (Helpful Ratio)...")


result_custom_info = db.command(
    "mapReduce", "reviews", 
    map=map_custom, 
    reduce=reduce_custom, 
    finalize=finalize_custom,
    out="mr_custom_results"
)

print("--- Job 3 Results (Sample of 5) ---")
for doc in db["mr_custom_results"].find().limit(5):
    print(doc)

Executing Job 3: Custom Logic (Helpful Ratio)...
--- Job 3 Results (Sample of 5) ---
{'_id': 'A85Y5VPQFAFL9', 'value': {'total_votes': 0.0, 'total_reviews': 1.0, 'avg_votes_per_review': 0.0}}
{'_id': 'A1KX1RGL86TKUR', 'value': {'total_votes': 0.0, 'total_reviews': 1.0, 'avg_votes_per_review': 0.0}}
{'_id': 'A10XX07AMFAFS6', 'value': {'total_votes': 0.0, 'total_reviews': 1.0, 'avg_votes_per_review': 0.0}}
{'_id': 'AP30SUWHUN2A5', 'value': {'total_votes': 0.0, 'total_reviews': 1.0, 'avg_votes_per_review': 0.0}}
{'_id': 'AGX5B9KIJX4EE', 'value': {'total_votes': 2.0, 'total_reviews': 1.0, 'avg_votes_per_review': 2.0}}
